## Generate MESH_output_streamflow.csv and Basin_average_water_balance.csv files: It reads MESH_input_streamflow.tb0 and MESH Flux Output Files in NetCDF 

In [1]:
import os
import geopandas as gpd
import xarray as xr
import pandas as pd
import numpy as np
from datetime import datetime


def combine_mesh_sim_obs(
    input_stations_comids: str,
    input_obs: str,
    input_ddb: str,
    mesh_flow: str,
    csv_filename: str
):
    """
    Combine observed and simulated MESH streamflow data for stations
    and save as a CSV file.

    Optimized for:
      - Graceful handling of missing COMIDs (fills with NaN)
      - High performance DataFrame construction
      - Robust file I/O and automatic path creation
      - Clean logging and safe merges

    Parameters
    ----------
    input_stations_comids : str
        Path to the stations COMIDs GeoPackage.
    input_obs : str
        Path to the observed streamflow file (TB0 format).
    input_ddb : str
        Path to the drainage database NetCDF.
    mesh_flow : str
        Path to the simulated streamflow NetCDF.
    csv_filename : str
        Output path for the combined CSV file.
    """

    print("🔹 Reading input files...")

    # --- Read drainage database (NetCDF) ---
    with xr.open_dataset(input_ddb) as db:
        segid = db["subbasin"].values

    # --- Read station COMIDs (GeoPackage) ---
    stations_comids = gpd.read_file(input_stations_comids, layer="points")
    stations_comids["Obs_NM"] = stations_comids["Obs_NM"].astype(str)

    # Sort stations: non-USGS first, then USGS
    station_ca = sorted(stations_comids.loc[stations_comids["SRC_obs"] != "USGS", "Obs_NM"].tolist())
    station_us = sorted(stations_comids.loc[stations_comids["SRC_obs"] == "USGS", "Obs_NM"].tolist())
    StationID = station_ca + station_us
    StationCOMID = [stations_comids.loc[stations_comids["Obs_NM"] == s, "COMID"].values[0] for s in StationID]

    # --- Match COMIDs to segid ---
    print("🔹 Matching station COMIDs with segid array...")
    missing = []
    for s, b in zip(StationID, StationCOMID):
        if not np.any(segid == b):
            missing.append((s, b))

    if missing:
        print(f"⚠️ {len(missing)} COMIDs not found in segid:")
        for s, b in missing:
            print(f"   - Station {s} (COMID {b}) missing. Will fill with NaN.")
    else:
        print("✅ All station COMIDs found in segid.")

    # --- Read simulated streamflow ---
    print("🔹 Reading simulated streamflow from:", mesh_flow)
    with xr.open_dataset(mesh_flow) as meshout:
        flow = meshout["QO"].values  # Adjust variable name if needed
        model_start_time = pd.to_datetime(str(meshout.time.values[0])).normalize()

    # --- Build simulated DataFrame efficiently ---
    print("🔹 Constructing simulated streamflow DataFrame...")

    sim_data = {}
    for s, b in zip(StationID, StationCOMID):
        idx = np.where(segid == b)[0]
        if len(idx) > 0:
            sim_data[f"QOSIM_{s}"] = flow[:, idx[0]]
        else:
            sim_data[f"QOSIM_{s}"] = np.full(flow.shape[0], np.nan)

    df_sim = pd.DataFrame(sim_data)
    df_sim.insert(0, "Dates", pd.date_range(start=model_start_time, periods=len(df_sim), freq="D"))

    # --- Read observed streamflow ---
    print("🔹 Reading observed streamflow from:", input_obs)
    with open(input_obs) as f:
        start_line = next(line for line in f if line.strip().startswith(":StartTime"))
    observed_start_time = datetime.strptime(
        " ".join(start_line.split()[1:]),
        "%Y/%m/%d %H:%M:%S.%f"
    )

    df_obs = pd.read_csv(input_obs, delimiter=r"\s+", skiprows=37, header=None)
    df_obs.columns = [f"QOMEAS_{s}" for s in StationID]
    df_obs.insert(0, "Dates", pd.date_range(start=observed_start_time, periods=len(df_obs), freq="D"))

    # --- Merge observed and simulated ---
    print("🔹 Merging observed and simulated data...")
    df_list = []

    for s in StationID:
        obs_col = f"QOMEAS_{s}"
        sim_col = f"QOSIM_{s}"

        if sim_col not in df_sim.columns:
            print(f"⚠️ No simulated data for {s}, skipping merge.")
            continue

        obs = df_obs[["Dates", obs_col]].copy()
        sim = df_sim[["Dates", sim_col]].copy()

        # Align by overlapping dates
        start = max(obs["Dates"].min(), sim["Dates"].min())
        end = min(obs["Dates"].max(), sim["Dates"].max())

        obs_sub = obs[(obs["Dates"] >= start) & (obs["Dates"] <= end)]
        sim_sub = sim[(sim["Dates"] >= start) & (sim["Dates"] <= end)]

        merged = pd.merge(obs_sub, sim_sub, on="Dates", how="inner")
        if merged.empty:
            print(f"⚠️ No overlapping period for station {s}, skipping.")
            continue

        merged.insert(0, "YEAR", merged["Dates"].dt.year)
        merged.insert(1, "JDAY", merged["Dates"].dt.dayofyear)
        merged.drop(columns=["Dates"], inplace=True)

        df_list.append(merged)

    if not df_list:
        raise ValueError("❌ No stations had both observed and simulated data to merge.")

    df_combined = pd.concat(df_list, axis=1)
    df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()]

    # --- Save to CSV ---
    os.makedirs(os.path.dirname(csv_filename), exist_ok=True)
    df_combined.to_csv(csv_filename, index=False)
    print(f"✅ Combined CSV saved to: {csv_filename}")

### Prepare and write: MESH_output_streamflow.csv files

In [ ]:
import os
import geopandas as gpd
import xarray as xr
import pandas as pd
import numpy as np
from datetime import datetime

def combine_mesh_sim_obs(
    input_stations_comids: str,
    input_obs: str,
    input_ddb: str,
    mesh_flow: str,
    csv_filename: str
):
    """
    Combine observed and simulated MESH streamflow data for stations
    and save as a CSV.

    Parameters
    ----------
    input_stations_comids : str
        File path to the stations COMIDs GeoPackage.
    input_obs : str
        File path to the observed streamflow file (TB0 format).
    input_ddb : str
        File path to the drainage database NetCDF.
    mesh_flow : str
        File path to the simulated streamflow NetCDF.
    csv_filename : str
        File path to save the combined CSV.
    """
    
    # --- Read input NetCDF drainage database ---
    db = xr.open_dataset(input_ddb)
    segid = db.variables['subbasin'].values
    db.close()

    # --- Extract the Station IDs for all stations ---
    stations_comids = gpd.read_file(input_stations_comids, layer="points")
    stations_comids["Obs_NM"] = stations_comids["Obs_NM"].astype(str)
    #StationID = sorted(stations_comids['Obs_NM'].dropna().unique().tolist()) 
    # Non-USGS stations first, sorted alphabetically
    station_ca = sorted(stations_comids.loc[stations_comids["SRC_obs"] != "USGS", "Obs_NM"].tolist())
    # USGS stations next, sorted alphabetically
    station_us = sorted(stations_comids.loc[stations_comids["SRC_obs"] == "USGS", "Obs_NM"].tolist())
    # Combine
    StationID = station_ca + station_us
    # --- Reorder COMIDs to match StationID ---
    StationCOMID = [stations_comids.loc[stations_comids["Obs_NM"] == s, "COMID"].values[0] for s in StationID]

    # --- Find indices of station COMIDs in segid ---
    #indices = [np.where(segid == b)[0][0] for b in StationCOMID]
    #indices = [np.where(segid == b)[0][0] if np.any(segid == b) else np.nan for b in StationCOMID]
##############################################################################
    indices = []
    missing_comids = []

  #  print("🔍 Checking COMID matches between stations and segid...")

    for b in StationCOMID:
        idx = np.where(segid == b)[0]
        if len(idx) > 0:
            indices.append(idx[0])
    #        print(f"✅ Found COMID {b} at index {idx[0]}")
        else:
            print(f"⚠️  COMID {b} not found in segid array.")
            missing_comids.append(b)

    if missing_comids:
        print("\n🚫 Missing COMIDs (not found in segid):")
        print(missing_comids)
   # else:
   #     print("\n✅ All station COMIDs found in segid.")

###############################################################################
    # --- Read simulated streamflow ---
    meshout = xr.open_dataset(mesh_flow)
    flow = meshout.variables['QO'].values
    model_start_times = meshout.time.values[0].astype('M8[ms]').astype('O').replace(microsecond=0)
    meshout.close()

    #df_sim = pd.DataFrame(flow[:, indices], columns=['QOSIM_' + col for col in StationID])
####################################################################
    df_sim = pd.DataFrame(index=range(flow.shape[0]))

    for s, b in zip(StationID, StationCOMID):
        idx = np.where(segid == b)[0]
        if len(idx) > 0:
            df_sim['QOSIM_' + s] = flow[:, idx[0]]
        else:
            df_sim['QOSIM_' + s] = np.nan
            print(f"⚠️ Filled NaN for station {s} (COMID {b} missing in segid).")

    sim_date = pd.DataFrame({'Dates': pd.date_range(start=model_start_times, periods=len(df_sim), freq='D')})
    df_sim = sim_date.join(df_sim)
###############################################################################
    #sim_date = pd.DataFrame({'Dates': pd.date_range(start=model_start_times, periods=len(df_sim), freq='D')})
    #df_sim = sim_date.join(df_sim)

    # --- Read observed streamflow ---
    df_obs = pd.read_csv(input_obs, delimiter=r'\s+', skiprows=37, header=None)
    df_obs.columns = ['QOMEAS_' + col for col in StationID]

    observed_start_time = datetime.strptime(
        ' '.join(next(line for line in open(input_obs) if line.strip().startswith(":StartTime")).split()[1:]),
        "%Y/%m/%d %H:%M:%S.%f"
    )
    obs_date = pd.DataFrame({'Dates': pd.date_range(start=observed_start_time, periods=len(df_obs), freq='D')})
    df_obs = obs_date.join(df_obs)

    # --- Merge observed and simulated data for each station ---
    df_list = []

    for station in StationID:
        obs = df_obs[['Dates', 'QOMEAS_' + station]].copy()
        sim = df_sim[['Dates', 'QOSIM_' + station]].copy()

        obs['Dates'] = pd.to_datetime(obs['Dates'])
        sim['Dates'] = pd.to_datetime(sim['Dates'])

        start = max(obs['Dates'].min(), sim['Dates'].min())
        end = min(obs['Dates'].max(), sim['Dates'].max())

        obs_sub = obs[(obs['Dates'] >= start) & (obs['Dates'] <= end)]
        sim_sub = sim[(sim['Dates'] >= start) & (sim['Dates'] <= end)]

        merged = pd.merge(obs_sub, sim_sub, on='Dates', how='inner')

        # Add YEAR and JDAY columns
        merged.insert(0, 'YEAR', merged['Dates'].dt.year)
        merged.insert(1, 'JDAY', merged['Dates'].dt.dayofyear)
        merged.drop(columns=['Dates'], inplace=True)

        df_list.append(merged)

    df_combined = pd.concat(df_list, axis=1)
    df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()]

    # --- Save to CSV ---
    df_combined.to_csv(csv_filename, index=False)
    print(f"✅ CSV file '{csv_filename}' has been saved.")


# --- Example usage ---
# combine_mesh_sim_obs(
#     input_stations_comids='D:\\Zelalem\\WSC\\combined_discharge_stations_comids.gpkg',
#     input_obs='D:\\Zelalem\\WSC\\MESH_input_streamflow_latlon.tb0',
#     input_ddb='D:\\Zelalem\\RUNs\\MESH_drainage_database_Polish_0p05_0p02_0p01.nc',
#     mesh_flow='D:\\Zelalem\\RUNs\\MESH_CaSRv2p1\\Average_GRU_Params\\QO_D_GRD.nc',
#     csv_filename='D:\\Zelalem\\RUNs\\MESH_CaSRv2p1\\Average_GRU_Params\\MESH_output_streamflow.csv'
# )

In [2]:
# Top-level folders
mesh_versions = ["Test"] #, "MESH_CaSRv2p1", "MESH_CaSRv3p1"]
gru_types = ["Average_GRU_Params", "Distributed_GRU_Params"]

# Base paths
base_path = r"D:\Zelalem\RUNs"
stations_comids_path = r"D:\Zelalem\WSC\combined_discharge_stations_comids.gpkg"
obs_path = r"D:\Zelalem\WSC\MESH_input_streamflow_latlon.tb0"
ddb_path = r"D:\Zelalem\RUNs\MESH_drainage_database_Polish_0p05_0p02_0p01.nc"

for mesh_ver in mesh_versions:
    for gru in gru_types:
        # Path to GRU folder
        gru_path = os.path.join(base_path, mesh_ver, gru)
        
        # Skip if the GRU folder does not exist
        if not os.path.exists(gru_path):
            print(f"⚠️ Skipping missing folder: {gru_path}")
            continue
        
        # Dynamically list subfolders inside this GRU folder
        mesh_subfolders = [f for f in os.listdir(gru_path) if os.path.isdir(os.path.join(gru_path, f))]
        
        for sub in mesh_subfolders:
            # Construct full paths
            mesh_flow_path = os.path.join(gru_path, sub, "QO_D_GRD.nc")
            csv_output_path = os.path.join(gru_path, sub, "MESH_output_streamflow.csv")
            
            # Skip if the QO_D_GRD.nc file does not exist
            if not os.path.exists(mesh_flow_path):
                print(f"⚠️ Skipping missing mesh flow file: {mesh_flow_path}")
                continue
            
            print(f"Running for: {mesh_ver} / {gru} / {sub}")
            print(f"Mesh flow: {mesh_flow_path}")
            print(f"CSV output: {csv_output_path}")
            
            # Call the function
            combine_mesh_sim_obs(
                input_stations_comids=stations_comids_path,
                input_obs=obs_path,
                input_ddb=ddb_path,
                mesh_flow=mesh_flow_path,
                csv_filename=csv_output_path
            )    

Running for: Test / Average_GRU_Params / MESH_1860mezt_full
Mesh flow: D:\Zelalem\RUNs\Test\Average_GRU_Params\MESH_1860mezt_full\QO_D_GRD.nc
CSV output: D:\Zelalem\RUNs\Test\Average_GRU_Params\MESH_1860mezt_full\MESH_output_streamflow.csv
🔹 Reading input files...
🔹 Matching station COMIDs with segid array...
⚠️ 1 COMIDs not found in segid:
   - Station 12448990 (COMID 78014226.0) missing. Will fill with NaN.
🔹 Reading simulated streamflow from: D:\Zelalem\RUNs\Test\Average_GRU_Params\MESH_1860mezt_full\QO_D_GRD.nc
🔹 Constructing simulated streamflow DataFrame...
🔹 Reading observed streamflow from: D:\Zelalem\WSC\MESH_input_streamflow_latlon2.tb0
🔹 Merging observed and simulated data...
✅ Combined CSV saved to: D:\Zelalem\RUNs\Test\Average_GRU_Params\MESH_1860mezt_full\MESH_output_streamflow.csv
Running for: Test / Average_GRU_Params / MESH_1p5p5_full
Mesh flow: D:\Zelalem\RUNs\Test\Average_GRU_Params\MESH_1p5p5_full\QO_D_GRD.nc
CSV output: D:\Zelalem\RUNs\Test\Average_GRU_Params\MESH_

### Prepare and write: Basin_average.csv files for various given variables

In [ ]:
### load modules
import os
import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd
import netCDF4 as nc
import matplotlib.pyplot as plt
import datetime
from datetime import datetime

In [ ]:
## Define inputs file path
input_ddb = 'D:\\Zelalem\\RUNs\\MESH_drainage_database_Polish_0p05_0p02_0p01.nc'
mesh_prec  = 'K:\\MESH_model_run\\OBASINAVG_CaSRv2p1\\PREC_D_GRD.nc'
mesh_rof  = 'K:\\MESH_model_run\\OBASINAVG_CaSRv2p1\\ROF_D_GRD.nc'
mesh_evap  = 'K:\\MESH_model_run\\OBASINAVG_CaSRv2p1\\ET_D_GRD.nc'
mesh_ta  = 'K:\\MESH_model_run\\OBASINAVG_CaSRv2p1\\TA_D_GRD.nc'

### Read input db netcdf files
db = xr.open_dataset(input_ddb)
db.close()

# Read the inputs from the drainage database files to calculate the drainage area
df_wb = pd.DataFrame()
df_wb['Rank'] = db.variables['Rank'].values
df_wb['Next'] = db.variables['Next'].values
df_wb['DA'] = db.variables['GridArea'].values

# Read and load the MESH outputs
meshout = xr.open_dataset(mesh_prec)
meshout.close()
prec = meshout.variables['PREC'].values
#
meshout = xr.open_dataset(mesh_evap)
meshout.close()
evap = meshout.variables['ET'].values
#
meshout = xr.open_dataset(mesh_ta)
meshout.close()
ta = meshout.variables['Ta'].values


# Ensure the shapes of 'prec' and 'GridArea' align before multiplication
prec = pd.DataFrame(prec) * db.variables['GridArea'].values
# Join the DataFrame with the PREC values
df_wb = df_wb.join(prec.T)

# Loop through the unique Rank values and calculate DA and the areal sum of the given variables 'water balance components' 
for i in df_wb['Rank'].unique().tolist():
    # Find the index where Rank matches the current value
    rank_index = df_wb[df_wb['Rank'] == i].index[0]
    # Find the areal sum of the given variables 'water balance components' where the 'Next' value matches the current Rank
    da_sum = df_wb.loc[df_wb['Next'] == i, 'DA':].sum(axis=0)
    df_wb.loc[rank_index, 'DA':] += da_sum
    
# Perform the division by the drainage area
df_wb.iloc[:, df_wb.columns.get_loc('DA') + 1:] = df_wb.iloc[:, df_wb.columns.get_loc('DA') + 1:].div(df_wb['DA'], axis=0)

### Selecte the water balance components for the stations of interest
# Loop through all columns from both DataFrames
df_sim = df_wb.iloc[indices, 3:].T
df_sim.columns = StationID
df_sim.columns = ['PREC_' + col for col in df_sim.columns]
df = df_date.copy()
df[df_sim.columns] = df_sim[df_sim.columns]

# Save the DataFrame to Basin_average_water_balance_DefaultRun.csv file
csv_filename = 'GL_less_5p_areadiff_Basin_average_water_balance.csv'
df.to_csv(os.path.join(directory, csv_filename), index=False, sep=',')
print(f"CSV file '{csv_filename}' has been saved.")   

In [ ]:
## Potential substitute for the above loop
# Loop through the unique Rank values and calculate DA and the areal sum of the given variables 'water balance components' 
for i in df_wb['Rank'].unique():
    # Find the index where Rank matches the current value
    rank_index = df_wb[df_wb['Rank'] == i].index[0]
    # Find the areal sum of the given variables 'water balance components' where the 'Next' value matches the current Rank
    da_sum = df_wb.loc[df_wb['Next'] == i, 'DA':].sum(axis=0)
    df_wb.loc[rank_index, 'DA':] += da_sum
    
# Perform the division by the drainage area
df_wb.iloc[:, df_wb.columns.get_loc('DA') + 1:] = df_wb.iloc[:, df_wb.columns.get_loc('DA') + 1:].div(df_wb['DA'], axis=0)

In [ ]:
## Define the start and end dates for the date range
start_date = '1980-10-01'
end_date = '2018-09-30'

## Create a date range from start_date to end_date
dates = pd.date_range(start=start_date, end=end_date, freq='D')

## Create a DataFrame with the date column
year = dates.year
julian_day = dates.dayofyear 

## Create a DataFrame with the year and Julian day columns
df_date = pd.DataFrame({'YEAR': year, 'JDAY': julian_day})
#df = pd.DataFrame({'Date': dates})

# Read MESH simulated streamflow for selected station locations from the NetCDF files
df_sim = pd.DataFrame(flow[:, indices], columns=StationID)
df_sim.columns = ['QOSIM_' + col for col in StationID]

# Create a list to store DataFrames in the desired order
df_list = [df_date]

# Append observed and simulated data for each station to the list
for station in StationID:
    df_list.append(df_obs[['QOMEAS_' + station]])
    df_list.append(df_sim[['QOSIM_' + station]])

# Concatenate all DataFrames in the list
df_combined = pd.concat(df_list, axis=1)

# Save the DataFrame to a CSV file
csv_filename = 'CaSRv3p1_CAMEL-spat_MESH_output_streamflow.csv'
df_combined.to_csv(os.path.join(directory, csv_filename), index=False)
print(f"CSV file '{csv_filename}' has been saved.")

In [ ]:
## Converting GRIPGL project observed and MESH simulated streamflow from netcdf file into csv file
import netCDF4 as nc
import pandas as pd
import os

def convert_netcdf_to_csv(netcdf_file, output_csv):
    # Open the NetCDF file
    ds = nc.Dataset(netcdf_file)
    
    # Extract the time variable
    time_var = ds.variables['time']

    # Extract the variables (adjust the variable names as needed)
    time = time_var[:]
    station_ids = ds.variables['station_id'][:]
    flow = ds.variables['Q'][:]

    # Get the units attribute and extract the reference date
    time_units = time_var.units
    reference_date = time_units.split('since')[-1].strip()

    # Handle various time units
    if 'days' in time_units.lower():
        times = pd.to_datetime(time, unit='D', origin=pd.Timestamp(reference_date))
    elif 'seconds' in time_units.lower():
        times = pd.to_datetime(time, unit='s', origin=pd.Timestamp(reference_date))
    elif 'hours' in time_units.lower():
        times = pd.to_datetime(time, unit='h', origin=pd.Timestamp(reference_date))
    elif 'minutes' in time_units.lower():
        times = pd.to_datetime(time, unit='m', origin=pd.Timestamp(reference_date))
    elif 'months' in time_units.lower():
        times = pd.to_datetime(time, unit='M', origin=pd.Timestamp(reference_date))
    elif 'years' in time_units.lower():
        times = pd.to_datetime(time, unit='Y', origin=pd.Timestamp(reference_date))
    else:
        raise ValueError("Unsupported time units")

    # Transpose the flow array to match the shape
    flow_transposed = flow.T

    # Create a DataFrame and set the 'Date' column explicitly 
    df = pd.DataFrame(flow_transposed, index=times, columns=station_ids.tolist()) 
    df.reset_index(inplace=True) 
    df.rename(columns={'index': 'Date'}, inplace=True)
    
    # Save to CSV file without adding an extra index column 
    df.to_csv(output_csv, index=False)

In [ ]:
# Set the directory and save the result
phase_list = [1, 2, 3, 4]
obj_list = [1, 2]
#
for phase in phase_list:
    for obj in obj_list:
        # input nc file path
        netcdf_file = os.path.join(os.getcwd(), 'MESH_GRIPGL', f'mesh-class-raven_phase_{phase}_objective_{obj}.nc') 
        # Output CSV file path
        output_csv = os.path.join(os.getcwd(), 'MESH_GRIPGL', f'mesh-class-raven_phase_{phase}_objective_{obj}.csv') 
        convert_netcdf_to_csv(netcdf_file, output_csv)

In [ ]:
# Set the directory and save the result
obj_list = [1, 2, 11, 12]
#
for obj in obj_list:
    netcdf_file = os.path.join(os.getcwd(), 'MESH_GRIPGL', f'all_gauges_{obj}.nc')
    output_csv = os.path.join(os.getcwd(), 'MESH_GRIPGL', f'all_gauges_{obj}.csv')
    convert_netcdf_to_csv(netcdf_file, output_csv)

In [ ]:
# Merge the the observation and simulation data

import pandas as pd

def merge_csv_side_by_side(file1, file2, output_file):
    try:
        # Read the CSV files
        df1 = pd.read_csv(file1)
        df2 = pd.read_csv(file2)

        # Check if 'Date' column exists in both files
        if 'Date' not in df1.columns or 'Date' not in df2.columns:
            raise KeyError("'Date' column not found in one or both files.")

        # Convert 'Date' column to datetime
        df1['Date'] = pd.to_datetime(df1['Date'])
        df2['Date'] = pd.to_datetime(df2['Date'])

        # Define the start and end dates based on the date range in file1
        start_date = df1['Date'].min()
        end_date = df1['Date'].max()

        # Create a date range from start_date to end_date
        dates = pd.date_range(start=start_date, end=end_date, freq='D')

        # Create a DataFrame with the year and Julian day columns, ensuring they are integers
        df_date = pd.DataFrame({
            'YEAR': dates.year,
            'JDAY': dates.dayofyear
        }).fillna(0).astype(int)

        # Merge dataframes on 'Date'
        merged_df = pd.merge(df1, df2, on='Date', suffixes=('_QOMEAS', '_QOSIM'))

        # Extract YEAR and JDAY from the merged 'Date' column
        merged_df['YEAR'] = merged_df['Date'].dt.year
        merged_df['JDAY'] = merged_df['Date'].dt.dayofyear

        # Find common columns, excluding 'Date', 'YEAR', 'JDAY'
        common_columns = df1.columns.intersection(df2.columns).difference(['Date'])

        # Use the smaller set of common columns
        common_columns = common_columns[:min(len(df1.columns), len(df2.columns))]

        # Create lists to store new column order
        new_columns = ['YEAR', 'JDAY']
        for col in common_columns:
            new_columns.append(f'{col}_QOMEAS')
            new_columns.append(f'{col}_QOSIM')

        # Add columns that are not common
        new_columns.extend([f'{col}_QOMEAS' for col in df1.columns if col not in common_columns and col not in ['Date', 'YEAR', 'JDAY']])
        new_columns.extend([f'{col}_QOSIM' for col in df2.columns if col not in common_columns and col not in ['Date', 'YEAR', 'JDAY']])

        # Reorder the columns to match the new column order
        merged_df = merged_df[new_columns]

        # Save the merged DataFrame to a new CSV file
        merged_df.to_csv(output_file, index=False)
        print("Merge completed successfully!")

    except Exception as e:
        print(f"An error occurred: {e}")

# Example usage
#merge_csv_side_by_side('file1.csv', 'file2.csv', 'merged_output.csv')

In [ ]:
# Mereg two observation data file
import pandas as pd
# Read the CSV files into DataFrames
df1 = pd.read_csv(os.path.join(os.getcwd(), 'MESH_GRIPGL', f'all_gauges_1.csv'))
df2 = pd.read_csv(os.path.join(os.getcwd(), 'MESH_GRIPGL', f'all_gauges_2.csv'))
out = os.path.join(os.getcwd(), 'MESH_GRIPGL', f'all_gauges_1_2.csv')
# Identify columns in df2 that are not in df1, except for 'Date'
columns_to_add = [col for col in df2.columns if col not in df1.columns or col == 'Date']
# Merge the DataFrames based on the 'Date' column, keeping only new columns from df2
df1 = df1.merge(df2[columns_to_add], on='Date', how='left')
# Save the merged DataFrame to a new CSV file
df1.to_csv(out, index=False)

In [ ]:
# Set the directory and save the result
phase_list = [1, 2]
obj_list = [1, 2]
#
for phase in phase_list:
    for obj in obj_list:
        # input nc file path
        file1 = os.path.join(os.getcwd(), 'MESH_GRIPGL', f'all_gauges_{obj}.csv')
        file2 = os.path.join(os.getcwd(), 'MESH_GRIPGL', f'mesh-class-raven_phase_{phase}_objective_{obj}.csv') 
        output_file = os.path.join(os.getcwd(), f'merged_mesh-class-raven_phase_{phase}_objective_{obj}.csv')
        merge_csv_side_by_side(file1, file2, output_file)

In [ ]:
#
phase_list = [3, 4]
obj_list = [1, 2]
#
for phase in phase_list:
    for obj in obj_list:
        # input nc file path
        file1 = os.path.join(os.getcwd(), 'MESH_GRIPGL', f'all_gauges_1{obj}.csv')
        file2 = os.path.join(os.getcwd(), 'MESH_GRIPGL', f'mesh-class-raven_phase_{phase}_objective_{obj}.csv') 
        output_file = os.path.join(os.getcwd(), f'merged_mesh-class-raven_phase_{phase}_objective_{obj}.csv')
        merge_csv_side_by_side(file1, file2, output_file)

In [ ]:
## Define inputs file path
directory = 'K:\\NAS_Project\\NAS_Model_Runs\\'
input_ddb = 'K:\\NAS_Project\\NAS_Model_Runs\\Network_topology_merit_with_lake.nc'
mesh_flow  = 'K:\\NAS_Project\\NAS_Model_Runs\\input_forcing.nc'

## mesh_ta  = '/home/zelalem/github-repos/TA_D_GRD.nc'
##
## Define the start and end dates for the date range
start_date = '1980-10-01'
end_date = '2018-09-30'

StationCOMID = [71007531, 72045942, 82017077, 72046302]
StationID = ['06CD002', '05BH004', '10ED002', '01AK009']

StationCOMID = [71027942]
StationID = ['06CD002']

In [ ]:
## Create a date range from start_date to end_date
dates = pd.date_range(start=start_date, end=end_date, freq='D')

## Create a DataFrame with the date column
year = dates.year
julian_day = dates.dayofyear 

## Create a DataFrame with the year and Julian day columns
df_date = pd.DataFrame({'YEAR': year, 'JDAY': julian_day})
#df = pd.DataFrame({'Date': dates})

### Read input db netcdf files
db = xs.open_dataset(input_ddb)
db.close()
#segid = db.variables['subbasin'].values
segid = db.variables['ID'].values

# Find the indices of station COMID's elements in segid: identify the rank of the station COMID
indices = [np.where(segid == b)[0][0] for b in StationCOMID]

In [ ]:
## Read input mesh output files in NetCDF
meshout = xs.open_dataset(mesh_flow)
meshout.close()
#prec = meshout.variables['RFF'].values
prec = meshout.variables['runoff'].values

# Read the inputs from the drainage database files to calculate the drainage area
df_wb = pd.DataFrame()
#df_wb['Rank'] = db.variables['Rank'].values
#df_wb['Next'] = db.variables['Next'].values
#df_wb['DA'] = db.variables['GridArea'].values

df_wb['ID'] = db.variables['ID'].values
df_wb['ID_next'] = db.variables['ID_next'].values
df_wb['DA'] = db.variables['area'].values

# Ensure the shapes of 'prec' and 'GridArea' align before multiplication
#prec = pd.DataFrame(prec) * db.variables['GridArea'].values
prec = pd.DataFrame(prec) * db.variables['area'].values

# Join the DataFrame with the PREC values
df_wb = df_wb.join(prec.T) 

In [ ]:
# Loop through the unique Rank values and calculate DA and the areal sum of the given variables 'water balance components' 
for i in df_wb['ID'].unique().tolist():
    # Find the index where Rank matches the current value
    rank_index = df_wb[df_wb['ID'] == i].index[0]
    # Find the areal sum of the given variables 'water balance components' where the 'Next' value matches the current Rank
    da_sum = df_wb.loc[df_wb['ID_next'] == i, 'DA':].sum(axis=0)
    df_wb.loc[rank_index, 'DA':] += da_sum
    
# Perform the division by the drainage area
df_wb.iloc[:, df_wb.columns.get_loc('DA') + 1:] = df_wb.iloc[:, df_wb.columns.get_loc('DA') + 1:].div(df_wb['DA'], axis=0)

In [ ]:
def update_rank_with_areal_sum_flow(df_wb):
    # Step 1: Get only DA and later columns
    da_cols = df_wb.columns[df_wb.columns.get_loc('DA'):]

    # Step 2: Aggregate DA columns based on flow direction (Rank -> Next)
    da_sums = df_wb.groupby('Next')[da_cols].sum().reset_index()

    # Step 3: For each Rank, sum values from rows where 'Next' == Rank
    rank_first = df_wb[~df_wb.duplicated('Rank')][['Rank']].copy()
    rank_first['row_index'] = rank_first.index

    # Step 4: Merge the sum with Rank flow direction
    merged = rank_first.merge(da_sums, left_on='Rank', right_on='Next', how='inner')

    # Step 5: Add aggregated DA values to corresponding rows based on 'Rank'
    df_wb.loc[merged['row_index'], da_cols] += merged[da_cols].values

    return df_wb

# Example usage
df_wb = update_rank_with_areal_sum_fast(df_wb)

In [ ]:
# Perform the division by the drainage area
# df_wb.iloc[:, df_wb.columns.get_loc('DA') + 1:] = df_wb.iloc[:, df_wb.columns.get_loc('DA') + 1:].div(df_wb['DA'], axis=0)

### Selecte the water balance components for the stations of interest
# Loop through all columns from both DataFrames
df_sim = df_wb.iloc[indices, 3:].T
df_sim.columns = StationID
df_sim.columns = ['RFF_' + col for col in df_sim.columns]
df = df_date.copy()
df[df_sim.columns] = df_sim[df_sim.columns]

# Save the DataFrame to Basin_average_water_balance_Default_Run.csv file   
csv_filename = 'Calg_Mizu_RFF_Q.csv'
df.to_csv(os.path.join(directory, csv_filename), index=False, sep=',')
print(f"CSV file '{csv_filename}' has been saved.")  

In [ ]:
df_wb.iloc[indices, 1:3].T

In [ ]:
import xarray as xr
import dask.array as da
import numpy as np
import pandas as pd

def extract_timeseries_by_subbasin(nc_path, variable_name, target_subbasin=None, target_latlon=None, save_csv=True):
    """
    Extract a time series for a specific subbasin by index or nearest lat/lon.

    Parameters:
        nc_path (str): Path to NetCDF file
        variable_name (str): Name of the variable to extract
        target_subbasin (int): Index of the subbasin (0-based)
        target_latlon (tuple): (lon, lat) to find the nearest subbasin
        save_csv (bool): Whether to save the result as a CSV

    Returns:
        pd.DataFrame: DataFrame with time series for the selected subbasin
    """
    ds = xr.open_dataset(nc_path)

    if 'subbasin' not in ds.dims:
        raise KeyError("No 'subbasin' dimension found in the dataset.")

    # Find subbasin by lat/lon if needed
    if target_latlon is not None:
        target_lon, target_lat = target_latlon
        lon_vals = ds['longitude'].values
        lat_vals = ds['latitude'].values

        distances = np.sqrt((lon_vals - target_lon) ** 2 + (lat_vals - target_lat) ** 2)
        target_subbasin = int(np.argmin(distances))
        print(f"Nearest subbasin index to (lon={target_lon}, lat={target_lat}) is {target_subbasin}")

    elif target_subbasin is None:
        raise ValueError("You must provide either a subbasin index or a lat/lon location.")

    if variable_name not in ds.variables:
        raise KeyError(f"Variable '{variable_name}' not found in the dataset.")

    # Extract data for the subbasin
    data = ds[variable_name].isel(subbasin=target_subbasin).load()  # Load just one subbasin

    # Combine time and data into DataFrame
    df = pd.DataFrame({
        'time': ds['time'].values,
        variable_name: data.values
    })

    if save_csv:
        csv_name = f"subbasin_{target_subbasin}_{variable_name}.csv"
        df.to_csv(csv_name, index=False)
        print(f"Saved to {csv_name}")

    return df

In [ ]:
# === Example usage ===
if __name__ == "__main__":
    nc_file = 'K:\\NAS_Project\\NAS_Model_Runs\\MESH_Run_0p0\\MESH_input_CanTrans_CaSR_1980_2018.nc'
    # variable = "CaSR_v3.1_P_TT_09975"
    variable = "RDRS_v2.1_P_TT_09944"

    # Option 1: extract by index
    subbasin_index = 41841
    df = extract_timeseries_by_subbasin(nc_file, variable_name=variable, target_subbasin=subbasin_index, save_csv=True)

    # Option 2: extract by nearest lat/lon
    # df = extract_timeseries_by_subbasin(nc_file, variable_name=variable, target_latlon=(106.3, 56.2), save_csv=True)

    print(df.head())